# Notebook 12 – Business Insights
## Converting EDA Results into Meaningful Business Observations

**Objective:** Move from **visualization → observation → insight → recommended action**.

This notebook demonstrates how to translate exploratory data analysis into useful business findings. It includes a sample sales dataset so the notebook can run immediately. Replace the sample data and column configuration with your real project dataset before reporting final conclusions.

### Key distinction
- **Observation:** What the data shows.
- **Insight:** What the observation may mean for the business.
- **Recommendation:** What action the business could consider.
- **Validation:** What additional information is needed before making a decision.




## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_colwidth", 140)
print("Libraries imported successfully.")

## 2. Load or create a dataset

The following sample data is for demonstration. To analyze your own data, replace the sample-data section with `df = pd.read_csv("your_dataset.csv")` and update the column configuration in Section 3.

In [ ]:
rng = np.random.default_rng(24)
n = 400

date = pd.date_range("2025-01-01", periods=n, freq="D")
category = rng.choice(["Category A", "Category B", "Category C"], size=n, p=[0.48, 0.34, 0.18])
region = rng.choice(["North", "South", "East", "West"], size=n)
channel = rng.choice(["Online", "Store"], size=n, p=[0.58, 0.42])
units = rng.integers(1, 8, size=n)
price_by_category = pd.Series(category).map({"Category A": 100, "Category B": 75, "Category C": 98}).to_numpy()
price = np.maximum(5, price_by_category + rng.normal(0, 12, size=n))
discount = rng.uniform(0, 0.25, size=n)
# Example pattern: Category C has fewer transactions but relatively high average selling price.
revenue = units * price * (1 - discount)
# Add a few unusually large values for anomaly-review practice.
revenue[rng.choice(n, size=4, replace=False)] *= 3

df = pd.DataFrame({
    "Date": date,
    "Category": category,
    "Region": region,
    "Channel": channel,
    "Units": units,
    "UnitPrice": price.round(2),
    "Discount": discount.round(3),
    "Revenue": revenue.round(2)
})

# For your own CSV dataset, use for example:
# df = pd.read_csv("your_dataset.csv")

display(df.head())
print("Dataset shape:", df.shape)

## 3. Configure business columns

Adjust these values to match your dataset. For example, a customer dataset may use `CustomerSegment` and `Profit`, while a retail dataset may use `Category` and `Sales`.

In [ ]:
DATE_COL = "Date"
CATEGORY_COL = "Category"
SEGMENT_COL = "Region"
CHANNEL_COL = "Channel"
SALES_COL = "Revenue"
QUANTITY_COL = "Units"
DISCOUNT_COL = "Discount"

def exists(col):
    return col in df.columns

def is_numeric(col):
    return exists(col) and pd.api.types.is_numeric_dtype(df[col])

def money(value):
    return f"{value:,.2f}"

required = [CATEGORY_COL, SEGMENT_COL, SALES_COL]
missing = [c for c in required if not exists(c)]
if missing:
    print("Update the column configuration. Missing columns:", missing)
else:
    print("Configuration looks ready.")

## 4. Validate data before drawing conclusions

Insights are only as reliable as the data. Check missing values, duplicate rows, data types, and basic numeric ranges before interpreting results.

In [ ]:
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing count"))

print("Completely duplicated rows:", int(df.duplicated().sum()))

print("Data types:")
display(df.dtypes.astype(str).to_frame("Data type"))

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)

## 5. Insight 1 — Important patterns: sales contribution by category

**Business question:** Which categories contribute most to total sales, and are some categories underperforming?

**How to move from chart to insight**
- **Observation:** Compare total sales and each category's percentage contribution.
- **Insight:** A category with a small sales contribution may deserve attention, but first compare its transaction count, price, and margin.
- **Potential action:** Investigate assortment, availability, promotion, and customer demand before changing investment.

In [ ]:
if exists(CATEGORY_COL) and is_numeric(SALES_COL):
    category_summary = df.groupby(CATEGORY_COL, dropna=False).agg(
        Total_Sales=(SALES_COL, "sum"),
        Average_Sale=(SALES_COL, "mean"),
        Transactions=(SALES_COL, "count")
    ).sort_values("Total_Sales", ascending=False)
    category_summary["Sales_Share_%"] = category_summary["Total_Sales"] / category_summary["Total_Sales"].sum() * 100
    display(category_summary.round(2))

    fig, ax = plt.subplots()
    category_summary["Total_Sales"].plot(kind="bar", ax=ax)
    ax.set_title("Total Sales by Category")
    ax.set_xlabel(CATEGORY_COL)
    ax.set_ylabel(f"Total {SALES_COL}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    top = category_summary.index[0]
    low = category_summary.index[-1]
    top_share = category_summary.loc[top, "Sales_Share_%"]
    low_share = category_summary.loc[low, "Sales_Share_%"]
    top_n = category_summary.loc[top, "Transactions"]
    low_n = category_summary.loc[low, "Transactions"]
    print("OBSERVATION:")
    print(f"{top} contributes the largest sales total ({top_share:.1f}% of recorded sales); "
          f"{low} contributes the smallest ({low_share:.1f}%).")
    print("INSIGHT TO INVESTIGATE:")
    print(f"{low} has {low_n} records versus {top_n} for {top}. Check whether the gap is mainly due to transaction volume, "
          "average order value, pricing, availability, or margin.")
    print("POTENTIAL ACTION:")
    print("Compare conversion, stock availability, margin, and repeat-purchase rates before deciding where to invest.")
else:
    print("Skipped: check CATEGORY_COL and numeric SALES_COL.")

## 6. Insight 2 — Trends over time

**Business question:** Are sales increasing, decreasing, or fluctuating over time?

A time trend is meaningful only if the date field is valid and observations can be ordered chronologically. Aggregating by month can reduce daily noise.

In [ ]:
if exists(DATE_COL) and is_numeric(SALES_COL):
    trend_df = df[[DATE_COL, SALES_COL]].copy()
    trend_df[DATE_COL] = pd.to_datetime(trend_df[DATE_COL], errors="coerce")
    trend_df = trend_df.dropna()
    if not trend_df.empty:
        monthly = trend_df.set_index(DATE_COL)[SALES_COL].resample("MS").sum()
        if len(monthly) >= 2:
            display(monthly.to_frame("Monthly_Sales").round(2))
            fig, ax = plt.subplots()
            monthly.plot(marker="o", ax=ax)
            ax.set_title("Monthly Sales Trend")
            ax.set_xlabel("Month")
            ax.set_ylabel(f"Total {SALES_COL}")
            plt.tight_layout()
            plt.show()
            change_pct = ((monthly.iloc[-1] - monthly.iloc[0]) / abs(monthly.iloc[0]) * 100
                          if monthly.iloc[0] != 0 else np.nan)
            print("OBSERVATION:")
            print(f"Sales in the first displayed month: {money(monthly.iloc[0])}; "
                  f"last displayed month: {money(monthly.iloc[-1])}.")
            print("INSIGHT TO INVESTIGATE:")
            if pd.notna(change_pct):
                print(f"First-to-last monthly sales changed by {change_pct:.1f}%. This is a comparison of endpoints, "
                      "not proof of a sustained trend.")
            print("POTENTIAL ACTION:")
            print("Check whether the pattern aligns with seasonality, marketing campaigns, stock availability, holidays, or reporting changes.")
        else:
            print("Not enough monthly periods to compare.")
    else:
        print("No valid date/sales rows available.")
else:
    print("Skipped: configure a valid date column and numeric sales column.")

## 7. Insight 3 — Relationships between variables

**Business question:** Is discount associated with sales value, or are quantity and unit price more closely related to revenue?

A scatter plot and correlation table can identify associations worth investigating. Correlation alone cannot show causation.

In [ ]:
relationship_cols = [c for c in [SALES_COL, QUANTITY_COL, DISCOUNT_COL] if is_numeric(c)]
if len(relationship_cols) >= 2:
    display(df[relationship_cols].corr().round(3))
    if is_numeric(DISCOUNT_COL) and is_numeric(SALES_COL):
        fig, ax = plt.subplots()
        sns.scatterplot(data=df, x=DISCOUNT_COL, y=SALES_COL, alpha=0.6, ax=ax)
        ax.set_title(f"{SALES_COL} vs {DISCOUNT_COL}")
        plt.tight_layout()
        plt.show()
        corr = df[[DISCOUNT_COL, SALES_COL]].dropna().corr().iloc[0, 1]
        print("OBSERVATION:")
        print(f"Pearson correlation between discount and sales value is {corr:.3f}.")
        print("INSIGHT TO INVESTIGATE:")
        print("A relationship may reflect differences in quantity, product price, customer mix, or promotional targeting.")
        print("POTENTIAL ACTION:")
        print("Evaluate discount effectiveness using profit margin, conversion, and a comparison group—not revenue alone.")
else:
    print("Skipped: at least two configured numeric columns are needed.")

## 8. Insight 4 — Anomalies and unusual values

**Business question:** Are there unusually high or low sales records that deserve validation?

The IQR method flags statistical outliers. It does not establish that a record is wrong.

In [ ]:
if is_numeric(SALES_COL):
    values = df[SALES_COL].dropna()
    if not values.empty:
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        anomaly_mask = (df[SALES_COL] < lower) | (df[SALES_COL] > upper)
        anomalies = df.loc[anomaly_mask].copy()
        print(f"IQR lower threshold: {lower:.2f}")
        print(f"IQR upper threshold: {upper:.2f}")
        print(f"Potential anomalies flagged: {len(anomalies)} ({len(anomalies) / len(df) * 100:.1f}% of rows)")
        display(anomalies.head(10))

        fig, ax = plt.subplots()
        sns.boxplot(x=values, ax=ax)
        ax.set_title(f"Potential Outliers in {SALES_COL}")
        plt.tight_layout()
        plt.show()

        print("OBSERVATION:")
        print(f"{len(anomalies)} record(s) fall outside the IQR thresholds.")
        print("INSIGHT TO INVESTIGATE:")
        print("These may be valid bulk orders, special events, data-entry mistakes, refunds, or unusual business cases.")
        print("POTENTIAL ACTION:")
        print("Validate flagged records against source transactions and document the reason before correcting or excluding any values.")
else:
    print("Skipped: selected sales measure is not numeric.")

## 9. Insight 5 — Interesting segments

**Business question:** Does sales performance differ across regions or customer segments?

Comparing both total sales and transaction counts helps distinguish a high-performing segment from one that simply has more records.

In [ ]:
if exists(SEGMENT_COL) and is_numeric(SALES_COL):
    segment_summary = df.groupby(SEGMENT_COL, dropna=False).agg(
        Total_Sales=(SALES_COL, "sum"),
        Average_Sale=(SALES_COL, "mean"),
        Transactions=(SALES_COL, "count")
    ).sort_values("Total_Sales", ascending=False)
    display(segment_summary.round(2))

    fig, ax = plt.subplots()
    sns.barplot(data=df, x=SEGMENT_COL, y=SALES_COL, estimator=np.mean, errorbar=None, ax=ax)
    ax.set_title(f"Average {SALES_COL} by {SEGMENT_COL}")
    ax.set_ylabel(f"Average {SALES_COL}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    best_total = segment_summary["Total_Sales"].idxmax()
    best_avg = segment_summary["Average_Sale"].idxmax()
    print("OBSERVATION:")
    print(f"{best_total} has the highest total sales, while {best_avg} has the highest average sale.")
    print("INSIGHT TO INVESTIGATE:")
    print("If these are different segments, one may lead in volume while another leads in average transaction value.")
    print("POTENTIAL ACTION:")
    print("Compare customer counts, costs, margin, acquisition cost, and retention before prioritizing a segment.")
else:
    print("Skipped: configure a segment and numeric sales column.")

## 10. Insight 6 — Potential business problems and opportunities

Use the following data checks as **signals for investigation**, not automatic diagnoses. The meaning depends on the business and dataset.

In [ ]:
checks = []

if is_numeric(SALES_COL):
    total_sales = df[SALES_COL].sum()
    zero_or_negative = int((df[SALES_COL] <= 0).sum())
    checks.append({
        "Signal": "Zero or negative sales values",
        "Evidence": f"{zero_or_negative} records",
        "Possible business question": "Are these refunds, cancellations, missing values, or valid adjustments?",
        "Suggested next step": "Inspect transaction type and source records."
    })

if is_numeric(DISCOUNT_COL):
    high_discount = int((df[DISCOUNT_COL] >= 0.20).sum())
    checks.append({
        "Signal": "High discounts (20% or more)",
        "Evidence": f"{high_discount} records",
        "Possible business question": "Are high discounts increasing conversion enough to justify the margin impact?",
        "Suggested next step": "Compare profit, conversion, and repeat purchase for discounted and non-discounted orders."
    })

if exists(CATEGORY_COL):
    counts = df[CATEGORY_COL].value_counts(dropna=False)
    rare = counts[counts < max(5, 0.02 * len(df))]
    checks.append({
        "Signal": "Rare categories",
        "Evidence": f"{len(rare)} categories below the chosen frequency threshold",
        "Possible business question": "Are these niche opportunities, low-demand products, or inconsistent category labels?",
        "Suggested next step": "Review category definitions, demand, profitability, and stock availability."
    })

if checks:
    display(pd.DataFrame(checks))
else:
    print("No checks could be run. Update the column configuration.")

## 11. Create a business insight register

Use this table to record findings after reviewing the real dataset. The template below is intentionally blank so conclusions are not invented.

In [ ]:
insight_register = pd.DataFrame(columns=[
    "Business question",
    "Observation (what the data shows)",
    "Insight (why it may matter)",
    "Potential business impact",
    "Recommended action",
    "Validation needed",
    "Priority"
])
display(insight_register)